In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from dotenv import load_dotenv
load_dotenv("/glazkov-dev/.env")

True

In [5]:
model_name = "Qwen/Qwen3-30B-A3B" #the smallest qwen with MOE
#don't fit into 40 Gb!

model_name = "Qwen/Qwen1.5-MoE-A2.7B-Chat" #no bridge in transformer_lens

In [6]:
if (False):
    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in float16 for speed
        bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
        # Normalized float 4-bit (optimal for LLMs)
        bnb_4bit_quant_type="nf4"
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map="cuda:3",
        cache_dir="/glazkov-dev/.cache",
    )

    # prepare the model input
    prompt = "Give me a short introduction to large language model."
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print("thinking content:", thinking_content)
    print("content:", content)


In [7]:

import torch
print(torch.__version__)
print(torch.version.cuda)


2.10.0+cu128
12.8


In [8]:
if (False):
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = "3" #without it gptq will try to start on all gpus
    os.environ.setdefault("HF_HUB_CACHE", "/glazkov-dev/.cache") #because doesnt work for TransformerBridge hf_config_overrides={"cache_dir": "/glazkov-dev/.cache"}
    from transformer_lens.model_bridge import TransformerBridge

    #bridge = TransformerBridge.boot_transformers("Qwen/Qwen3-30B-A3B-GPTQ-Int4", device="auto", hf_config_overrides={"cache_dir": "/glazkov-dev/.cache"})
    bridge = TransformerBridge.boot_transformers("Qwen/Qwen3-30B-A3B-GPTQ-Int4", device="cuda:0",)


Trying DeepseekV2ForCausalLM

In [9]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3" #этого требует transformer_lens внутри bridge кажется
import transformer_lens
print(transformer_lens.__file__)
from transformer_lens.model_bridge import TransformerBridge

MODEL = "deepseek-ai/DeepSeek-V2-Lite-Chat"
quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
        bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
        # Normalized float 4-bit (optimal for LLMs)
        bnb_4bit_quant_type="nf4",
        llm_int8_skip_modules=[
            "lm_head",
            "mlp.gate", #because DeepSeekV2ForCausalLM has a f.linear(self.gate.weight...) instead of self.gate()
            #it couldn't be substituded with Linear4bit
        ],
    )

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


/glazkov-dev/TransformerLens/transformer_lens/__init__.py


Loading so we will not have quantization applied!

In [10]:
if (False):
    bridge = TransformerBridge.boot_transformers(MODEL, 
                                             device="cuda:0",
                                             hf_config_overrides={
                                                 #"device": "cuda:3",
                                                 "quantization_config": quantization_config.to_dict(),
                                             },)

In [11]:
import gc
gc.collect()
torch.cuda.empty_cache()

If loading takes about 10 seconds - it loads on ONE GPU correctly. If it takes longer (1 min) - it probably tries to load on several gpu (ignoring device_map='cuda:0') and crashes with strange errors like "We encountered some issues during automatic conversion of the weights. For details look at the CONVERSION entries of the above report!"

In [12]:
from transformers import DeepseekV2ForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL, 
                                             trust_remote_code=False, #нужно в версии transformers 4, чтобы загружать доп код от дипсика, но в новых уже есть нативая поддержка DeepSeekV2 
                                                device_map='cuda:0',
                                                # dtype=torch.float16,
                                                quantization_config=quantization_config,
                                                # attn_implementation="sdpa", 
                                                # kv_cache_dtype="int8"
                                                )

Loading weights:   0%|          | 0/351 [00:00<?, ?it/s]

In [13]:
model

DeepseekV2ForCausalLM(
  (model): DeepseekV2Model(
    (embed_tokens): Embedding(102400, 2048)
    (layers): ModuleList(
      (0): DeepseekV2DecoderLayer(
        (self_attn): DeepseekV2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=3072, bias=False)
          (kv_a_proj_with_mqa): Linear4bit(in_features=2048, out_features=576, bias=False)
          (kv_a_layernorm): DeepseekV2RMSNorm((512,), eps=1e-06)
          (kv_b_proj): Linear4bit(in_features=512, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): DeepseekV2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=10944, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=10944, bias=False)
          (down_proj): Linear4bit(in_features=10944, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): DeepseekV2RMSNorm((2048,), eps=1e-06)
  

In [14]:
bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [15]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(102400, 2048)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): DeepseekV2RotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): MLABlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): DeepseekV2DecoderLayer(
        (self_attn): MLAAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.at

In [16]:
from utils.mmlu_batch_generator import MMLUBatchGenerator
SELECTED_TASKS = ["anatomy",
            # "conceptual_physics",
            # "human_sexuality",
            "machine_learning",
            "management",
            # "marketing",
            # "nutrition",
             "philosophy",
            # "us_foreign_policy",
              "world_religions"]
gen = MMLUBatchGenerator(subjects=SELECTED_TASKS, split='validation', batch_size=16, include_metadata=False)

Initialized MMLU batch generator with 5 subjects
Split: validation, Batch size: 16


In [17]:
from typing import List
@torch.no_grad()
def arbitrary_input(model: torch.nn.Module, tokenizer, input: List[str]):
    inputs = tokenizer(input, return_tensors="pt",
            padding=True,  # Pad to longest in batch
            truncation=True,
            max_length=512,  # Adjust based on your needs
            add_special_tokens=True
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model(**inputs)
    return outputs

In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=False,
                                                device_map='cuda:3',
                                                )

for batch in gen:
    inputs = tokenizer(batch, return_tensors="pt",
            padding=True,  # Pad to longest in batch
            truncation=True,
            max_length=512,  # Adjust based on your needs
            add_special_tokens=True
    )  
    inputs = {k: v.to(bridge.device) for k, v in inputs.items()}
    print(inputs.keys())
    bridge_outputs, cache = bridge.run_with_cache(inputs['input_ids'], attention_mask=inputs['attention_mask'])
    print(bridge_outputs)  # Example: (batch_size, sequence_length, vocab_size)
    break  # Remove this break to process all batches

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📚 Loading subject 1/5: anatomy
  ✓ Created batch with 14 questions
dict_keys(['input_ids', 'attention_mask'])
tensor([[[ 12.1250,  12.8750,   9.2500,  ...,   2.3438,   2.3594,   2.4531],
         [ 16.1250,  15.3125,  11.6250,  ...,   6.0312,   6.0000,   6.2188],
         [ 14.1250,  14.7500,  10.8125,  ...,   3.5312,   3.5469,   3.7031],
         ...,
         [ -3.3438,  -0.4492,  -8.3750,  ...,  -7.8750,  -7.5312,  -7.9375],
         [ -3.3438,  -0.4941,  -8.3750,  ...,  -7.8750,  -7.5000,  -7.9062],
         [ -3.4219,  -0.5078,  -8.4375,  ...,  -7.8125,  -7.4375,  -7.8438]],

        [[ 12.8125,  12.4375,   9.8125,  ...,   3.4531,   3.4688,   3.5938],
         [ 17.1250,  15.6250,  11.6250,  ...,   6.5000,   6.5000,   6.7188],
         [ 14.1250,  14.0000,  10.1250,  ...,   3.1562,   3.2031,   3.3438],
         ...,
         [ -3.1250,  -0.6719,  -8.3125,  ...,  -7.8750,  -7.5312,  -7.9375],
         [ -3.1562,  -0.6875,  -8.3125,  ...,  -7.8125,  -7.4375,  -7.8438],
         [ -

In [19]:
bridge_outputs.shape

torch.Size([14, 34, 102400])

In [20]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(102400, 2048)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): DeepseekV2RotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): MLABlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): DeepseekV2DecoderLayer(
        (self_attn): MLAAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.at

In [21]:
model.model.layers[1].mlp.gate

Linear(in_features=2048, out_features=64, bias=False)

In [22]:
inputs['input_ids'].shape

torch.Size([14, 34])

In [23]:
moe_bridge = bridge.blocks[1].mlp
native_moe = moe_bridge.original_component

print("MoE bridge:", type(moe_bridge))
print("Original MoE:", type(native_moe))

print("Router:")
print("  module:", type(native_moe.gate))
print("  weight:", native_moe.gate.weight.shape)
print("  dtype:", native_moe.gate.weight.dtype)

shared_bridge = moe_bridge.shared_experts
print("Shared bridge:", type(shared_bridge))
print("Original shared:", type(shared_bridge.original_component))

gate_bridge = shared_bridge.gate
print("Shared gate bridge:", type(gate_bridge))
print("Shared gate original:", type(gate_bridge.original_component))

MoE bridge: <class 'transformer_lens.model_bridge.generalized_components.moe.MoEBridge'>
Original MoE: <class 'transformers.models.deepseek_v2.modeling_deepseek_v2.DeepseekV2Moe'>
Router:
  module: <class 'torch.nn.modules.linear.Linear'>
  weight: torch.Size([64, 2048])
  dtype: torch.bfloat16
Shared bridge: <class 'transformer_lens.model_bridge.generalized_components.gated_mlp.GatedMLPBridge'>
Original shared: <class 'transformers.models.deepseek_v2.modeling_deepseek_v2.DeepseekV2MLP'>
Shared gate bridge: <class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>
Shared gate original: <class 'bitsandbytes.nn.modules.Linear4bit'>


In [24]:
for name in bridge.hook_dict:
    if "mlp" in name or "expert" in name:
        print(name)

blocks.0.hook_mlp_in
blocks.0.mlp.hook_in
blocks.0.mlp.hook_out
blocks.0.mlp.hook_router_scores
blocks.1.hook_mlp_in
blocks.1.mlp.hook_in
blocks.1.mlp.hook_out
blocks.1.mlp.hook_router_scores
blocks.1.mlp.shared_experts.hook_in
blocks.1.mlp.shared_experts.hook_out
blocks.1.mlp.shared_experts.gate.hook_in
blocks.1.mlp.shared_experts.gate.hook_out
blocks.1.mlp.shared_experts.in.hook_in
blocks.1.mlp.shared_experts.in.hook_out
blocks.1.mlp.shared_experts.out.hook_in
blocks.1.mlp.shared_experts.out.hook_out
blocks.2.hook_mlp_in
blocks.2.mlp.hook_in
blocks.2.mlp.hook_out
blocks.2.mlp.hook_router_scores
blocks.2.mlp.shared_experts.hook_in
blocks.2.mlp.shared_experts.hook_out
blocks.2.mlp.shared_experts.gate.hook_in
blocks.2.mlp.shared_experts.gate.hook_out
blocks.2.mlp.shared_experts.in.hook_in
blocks.2.mlp.shared_experts.in.hook_out
blocks.2.mlp.shared_experts.out.hook_in
blocks.2.mlp.shared_experts.out.hook_out
blocks.3.hook_mlp_in
blocks.3.mlp.hook_in
blocks.3.mlp.hook_out
blocks.3.mlp.hoo

In [25]:
for name in bridge.hook_dict:
        print(name)

embed.hook_in
embed.hook_out
rotary_emb.hook_in
rotary_emb.hook_out
rotary_emb.hook_cos
rotary_emb.hook_sin
blocks.0.hook_in
blocks.0.hook_out
blocks.0.hook_mlp_in
blocks.0.ln1.hook_in
blocks.0.ln1.hook_out
blocks.0.ln1.hook_normalized
blocks.0.ln1.hook_scale
blocks.0.ln2.hook_in
blocks.0.ln2.hook_out
blocks.0.ln2.hook_normalized
blocks.0.ln2.hook_scale
blocks.0.attn.hook_in
blocks.0.attn.hook_out
blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_pattern
blocks.0.attn.hook_hidden_states
blocks.0.attn.hook_result
blocks.0.attn.hook_rot_k
blocks.0.attn.hook_rot_q
blocks.0.attn.hook_cos
blocks.0.attn.hook_sin
blocks.0.attn.hook_q_latent
blocks.0.attn.hook_kv_latent
blocks.0.attn.hook_q
blocks.0.attn.hook_k
blocks.0.attn.hook_v
blocks.0.attn.q_proj.hook_in
blocks.0.attn.q_proj.hook_out
blocks.0.attn.kv_a_proj_with_mqa.hook_in
blocks.0.attn.kv_a_proj_with_mqa.hook_out
blocks.0.attn.kv_a_layernorm.hook_in
blocks.0.attn.kv_a_layernorm.hook_out
blocks.0.attn.kv_a_layernorm.hook_normalized
bloc

In [26]:
for k in cache.keys():
    print(k, cache[k].shape)

embed.hook_in torch.Size([14, 34])
embed.hook_out torch.Size([14, 34, 2048])
hook_embed torch.Size([14, 34, 2048])
rotary_emb.hook_in torch.Size([14, 34, 2048])
blocks.0.hook_in torch.Size([14, 34, 2048])
blocks.0.hook_resid_pre torch.Size([14, 34, 2048])
blocks.0.ln1.hook_in torch.Size([14, 34, 2048])
blocks.0.ln1.hook_scale torch.Size([14, 34, 1])
blocks.0.ln1.hook_normalized torch.Size([14, 34, 2048])
blocks.0.ln1.hook_out torch.Size([14, 34, 2048])
blocks.0.attn.hook_in torch.Size([14, 34, 2048])
blocks.0.attn.q_proj.hook_in torch.Size([14, 34, 2048])
blocks.0.attn.q_proj.hook_out torch.Size([14, 34, 3072])
blocks.0.attn.kv_a_proj_with_mqa.hook_in torch.Size([14, 34, 2048])
blocks.0.attn.kv_a_proj_with_mqa.hook_out torch.Size([14, 34, 576])
blocks.0.attn.kv_a_layernorm.hook_in torch.Size([14, 34, 512])
blocks.0.attn.kv_a_layernorm.hook_scale torch.Size([14, 34, 1])
blocks.0.attn.kv_a_layernorm.hook_normalized torch.Size([14, 34, 512])
blocks.0.attn.kv_a_layernorm.hook_out torch.Siz

In [27]:
cache['embed.hook_in'].shape #просто кэширует активации

torch.Size([14, 34])

In [ ]:
bridge.run_with_hooks()